# Download Socioeconomic Data for News Deserts Analysis

Downloads three datasets from **Danmarks Statistik's API** and merges them into `socioeconomic_data.csv`:

| Variable | DST Table | What I extract |
|---|---|---|
| Average income | `INDKP101` | Gennemsnit disponibel indkomst per kommune |
| Higher education share | `HFUDD10` | (H60 + H70) / TOT × 100 per kommune |
| Voter turnout | `KVRES` | Stemmeprocent per kommune (KV25 or KV21) |


In [118]:
import requests
import json
import pandas as pd
from pathlib import Path
from io import StringIO

MUNI_CSV = Path("dk_municipalities.csv")
OUTPUT_CSV = Path("socioeconomic_data.csv")

In [119]:
def dst_tableinfo(table_id: str) -> dict:
    """Print all variables and their valid codes for a DST table."""
    url = f"https://api.statbank.dk/v1/tableinfo/{table_id}?format=JSON"
    resp = requests.get(url)
    resp.raise_for_status()
    info = resp.json()

    print(f"Table: {info['id']} — {info['text']}")
    print("=" * 60)
    for var in info.get("variables", []):
        vals = var.get("values", [])
        print(f"\n  {var['id']} ({var['text']}) — {len(vals)} values")
        for v in vals[:10]:
            print(f"    {v['id']:>12s} : {v['text']}")
        if len(vals) > 10:
            print(f"    ...")
            for v in vals[-3:]:
                print(f"    {v['id']:>12s} : {v['text']}")
            print(f"    ({len(vals)} total)")
    return info


def dst_download_csv(table_id: str, params: dict) -> pd.DataFrame:
    """Download data from DST via POST. Prints DST's error on failure."""
    body = {
        "table": table_id,
        "format": "CSV",
        "delimiter": "Semicolon",
        "variables": [
            {"code": var_id, "values": codes}
            for var_id, codes in params.items()
        ],
    }
    resp = requests.post("https://api.statbank.dk/v1/data", json=body)

    if resp.status_code != 200:
        print(f"\n{'!'*60}")
        print(f"ERROR {resp.status_code} from DST API")
        print(f"{'!'*60}")
        print(f"\nDST says: {resp.text[:500]}")
        print(f"\nRequest body was:")
        print(json.dumps(body, indent=2, ensure_ascii=False))
        print(f"\n→ Fix: run dst_tableinfo('{table_id}') and compare variable names/codes.")
        resp.raise_for_status()

    df = pd.read_csv(StringIO(resp.text), sep=";")
    print(f"✓ Downloaded {table_id}: {len(df)} rows, cols: {list(df.columns)}")
    return df

In [120]:
dst_tableinfo("INDKP101")
dst_tableinfo("HFUDD10")
dst_tableinfo("KVRES")

Table: INDKP101 — Personer

  OMRÅDE (område) — 110 values
             000 : Hele landet
              01 : Landsdel Byen København
             101 : København
             147 : Frederiksberg
             155 : Dragør
             185 : Tårnby
              02 : Landsdel Københavns omegn
             165 : Albertslund
             151 : Ballerup
             153 : Brøndby
    ...
             787 : Thisted
             820 : Vesthimmerlands
             851 : Aalborg
    (110 total)

  ENHED (enhed) — 4 values
             101 : Personer med indkomsttypen (antal)
             110 : Indkomstbeløb (1.000 kr.)
             116 : Gennemsnit for alle personer (kr.)
             121 : Gennemsnit for personer med indkomsttypen (kr.)

  KOEN (køn) — 3 values
             MOK : Mænd og kvinder i alt
               M : Mænd
               K : Kvinder

  INDKOMSTTYPE (indkomsttype) — 39 values
             100 : 1 Disponibel indkomst (2+30-31-32-35)
             105 : 2 Indkomst i alt, før ska

{'id': 'KVRES',
 'text': 'Valg til kommunalbestyrelser',
 'description': 'Valg til kommunalbestyrelser efter område, valgresultat og tid',
 'unit': 'Antal',
 'suppressedDataValue': '0',
 'updated': '2022-04-07T08:00:00',
 'active': True,
 'contacts': [{'name': 'Dorthe Larsen',
   'phone': '+4523498326',
   'mail': 'dla@dst.dk'}],
 'documentation': {'id': 'c7836c78-b587-461a-96b6-af5648e390e5',
  'url': 'https://www.dst.dk/statistikdokumentation/c7836c78-b587-461a-96b6-af5648e390e5'},
 'footnote': None,
 'variables': [{'id': 'OMRÅDE',
   'text': 'område',
   'elimination': True,
   'time': False,
   'map': 'Denmark_municipality_07',
   'values': [{'id': '000', 'text': 'Hele landet'},
    {'id': '084', 'text': 'Region Hovedstaden'},
    {'id': '101', 'text': 'København'},
    {'id': '147', 'text': 'Frederiksberg'},
    {'id': '155', 'text': 'Dragør'},
    {'id': '185', 'text': 'Tårnby'},
    {'id': '165', 'text': 'Albertslund'},
    {'id': '151', 'text': 'Ballerup'},
    {'id': '153', 't

In [121]:
df_income_raw = dst_download_csv("INDKP101", {
    "OMRÅDE":        ["*"],      
    "KOEN":          ["MOK"],    
    "ENHED":         ["116"],    
    "INDKOMSTTYPE":  ["100"],     
    "Tid":           ["2024"],   
})
df_income_raw.head()

✓ Downloaded INDKP101: 110 rows, cols: ['OMRÅDE', 'KOEN', 'ENHED', 'INDKOMSTTYPE', 'TID', 'INDHOLD']


,OMRÅDE,KOEN,ENHED,INDKOMSTTYPE,TID,INDHOLD
0,Hele landet,Mænd og kvinder i alt,Gennemsnit for alle personer (kr.),1 Disponibel indkomst (2+30-31-32-35),2024,287682
1,Landsdel Byen København,Mænd og kvinder i alt,Gennemsnit for alle personer (kr.),1 Disponibel indkomst (2+30-31-32-35),2024,303697
2,København,Mænd og kvinder i alt,Gennemsnit for alle personer (kr.),1 Disponibel indkomst (2+30-31-32-35),2024,295836
3,Frederiksberg,Mænd og kvinder i alt,Gennemsnit for alle personer (kr.),1 Disponibel indkomst (2+30-31-32-35),2024,344810
4,Dragør,Mænd og kvinder i alt,Gennemsnit for alle personer (kr.),1 Disponibel indkomst (2+30-31-32-35),2024,375847


## 1. Income — HFUDD10


In [122]:
muni_lookup = pd.read_csv(MUNI_CSV)[["kommune_kode", "kommune_name"]]

candidate = df_income_raw.merge(
    muni_lookup,
    left_on="OMRÅDE",
    right_on="kommune_name",
    how="inner",
)

if len(candidate) == 0:
    df_income_raw = df_income_raw.assign(
        OMRÅDE_NAME=df_income_raw["OMRÅDE"].str.replace(r"^\d+\s+", "", regex=True)
    )
    candidate = df_income_raw.merge(
        muni_lookup,
        left_on="OMRÅDE_NAME",
        right_on="kommune_name",
        how="inner",
    )

income = (
    candidate
    .rename(columns={"INDHOLD": "avg_income"})
    [["kommune_kode", "avg_income"]]
    .assign(avg_income=lambda d: pd.to_numeric(d["avg_income"], errors="coerce"))
    .dropna(subset=["avg_income"])
    .reset_index(drop=True)
)

print(f"{len(income)} municipalities")
print(f"Mean avg income: {income['avg_income'].mean():,.0f} kr.")
print(f"Range: {income['avg_income'].min():,.0f} – {income['avg_income'].max():,.0f} kr.")
income.head()

98 municipalities
Mean avg income: 285,976 kr.
Range: 229,128 – 570,329 kr.


,kommune_kode,avg_income
0,101,295836
1,147,344810
2,155,375847
3,185,299878
4,165,248808



## 2. Education — HFUDD10



In [123]:
df_edu_raw = dst_download_csv("HFUDD10", {
    "BOPOMR":    ["*"],                     # All municipalities
    "HERKOMST":  ["TOT"],                   # All origins
    "HFUDD":     ["TOT", "H60", "H70"],    # Total + higher ed
    "ALDER":     ["TOT"],                   # All ages (15-69)
    "KØN":       ["TOT"],                   # Both sexes
    "Tid":       ["2019"],                  # Latest available
})
df_edu_raw.head()

✓ Downloaded HFUDD10: 315 rows, cols: ['BOPOMR', 'HERKOMST', 'HFUDD', 'ALDER', 'KØN', 'TID', 'INDHOLD']


,BOPOMR,HERKOMST,HFUDD,ALDER,KØN,TID,INDHOLD
0,Hele landet,I alt,I alt,Alder i alt,I alt,2019,4032807
1,Hele landet,I alt,"H60 Bacheloruddannelser, BACH",Alder i alt,I alt,2019,93407
2,Hele landet,I alt,"H70 Lange videregående uddannelser, LVU",Alder i alt,I alt,2019,400602
3,Region Hovedstaden,I alt,I alt,Alder i alt,I alt,2019,1303077
4,Region Hovedstaden,I alt,"H60 Bacheloruddannelser, BACH",Alder i alt,I alt,2019,49086


In [124]:
print("df_edu columns:", df_edu.columns.tolist())
print("Unique HFUDD values:", df_edu["HFUDD"].unique())
print()
print("Pivot columns:", pivot.columns.tolist())
print(pivot.head())

df_edu columns: ['BOPOMR', 'HERKOMST', 'HFUDD', 'ALDER', 'KØN', 'TID', 'INDHOLD', 'kommune_kode', 'kommune_name']
Unique HFUDD values: ['I alt' 'H60 Bacheloruddannelser, BACH'
 'H70 Lange videregående uddannelser, LVU']

Pivot columns: ['kommune_kode', 'H60 Bacheloruddannelser, BACH', 'H70 Lange videregående uddannelser, LVU', 'I alt']
HFUDD  kommune_kode  H60 Bacheloruddannelser, BACH  \
0               101                          30526   
1               147                           4514   
2               151                            433   
3               153                            340   
4               155                            179   

HFUDD  H70 Lange videregående uddannelser, LVU   I alt  
0                                       105604  484491  
1                                        19866   74853  
2                                         2946   32257  
3                                         1494   24251  
4                                         1218    90

In [125]:
print("df_edu_raw rows:", len(df_edu_raw))
print(df_edu_raw.head(10))
print()
print("Unique BOPOMR values (first 20):", df_edu_raw["BOPOMR"].unique()[:20])
print("Unique HFUDD in raw:", df_edu_raw["HFUDD"].unique())
print("Unique TID:", df_edu_raw["TID"].unique())
print()
print("After kommune_kode parse, df_edu rows:", len(df_edu))


df_edu_raw rows: 315
               BOPOMR HERKOMST                                    HFUDD  \
0         Hele landet    I alt                                    I alt   
1         Hele landet    I alt            H60 Bacheloruddannelser, BACH   
2         Hele landet    I alt  H70 Lange videregående uddannelser, LVU   
3  Region Hovedstaden    I alt                                    I alt   
4  Region Hovedstaden    I alt            H60 Bacheloruddannelser, BACH   
5  Region Hovedstaden    I alt  H70 Lange videregående uddannelser, LVU   
6           København    I alt                                    I alt   
7           København    I alt            H60 Bacheloruddannelser, BACH   
8           København    I alt  H70 Lange videregående uddannelser, LVU   
9       Frederiksberg    I alt                                    I alt   

         ALDER    KØN   TID  INDHOLD  
0  Alder i alt  I alt  2019  4032807  
1  Alder i alt  I alt  2019    93407  
2  Alder i alt  I alt  2019   400602

In [126]:
muni_lookup = pd.read_csv("dk_municipalities.csv")[["kommune_kode", "kommune_name"]]

df_edu = (
    df_edu_raw
    .drop(columns=["kommune_kode"], errors="ignore")
    .merge(muni_lookup, left_on="BOPOMR", right_on="kommune_name", how="inner")
)

print(f"After name-based join: {len(df_edu)} rows, "
      f"{df_edu['kommune_kode'].nunique()} unique municipalities")

# Pivot using the actual HFUDD labels Statbank returned
pivot = df_edu.pivot_table(
    index="kommune_kode",
    columns="HFUDD",
    values="INDHOLD",
    aggfunc="sum",
).reset_index()

print("Pivot columns:", pivot.columns.tolist())

def find_col(prefix_or_substring):
    matches = [c for c in pivot.columns if str(c).startswith(prefix_or_substring)
               or prefix_or_substring in str(c)]
    return matches[0] if matches else None

H60_COL = find_col("H60")
H70_COL = find_col("H70")
TOT_COL = find_col("I alt")  

missing = {"H60": H60_COL, "H70": H70_COL, "TOT": TOT_COL}
missing = [k for k, v in missing.items() if v is None]
if missing:
    raise KeyError(f"Could not find columns for: {missing}. "
                   f"Available: {pivot.columns.tolist()}")

print(f"Resolved: H60='{H60_COL}', H70='{H70_COL}', TOT='{TOT_COL}'")

higher = pd.to_numeric(pivot[H60_COL], errors="coerce").fillna(0) \
       + pd.to_numeric(pivot[H70_COL], errors="coerce").fillna(0)
total  = pd.to_numeric(pivot[TOT_COL], errors="coerce")

education = pd.DataFrame({
    "kommune_kode": pivot["kommune_kode"],
    "pct_higher_education": (higher / total * 100).round(2),
})

print(f"\n{len(education)} municipalities")
print(f"Mean higher ed share: {education['pct_higher_education'].mean():.1f}%")
print(f"Range: {education['pct_higher_education'].min():.1f}% – "
      f"{education['pct_higher_education'].max():.1f}%")
education.head()

After name-based join: 294 rows, 98 unique municipalities
Pivot columns: ['kommune_kode', 'H60 Bacheloruddannelser, BACH', 'H70 Lange videregående uddannelser, LVU', 'I alt']
Resolved: H60='H60 Bacheloruddannelser, BACH', H70='H70 Lange videregående uddannelser, LVU', TOT='I alt'

98 municipalities
Mean higher ed share: 8.8%
Range: 2.6% – 32.6%


,kommune_kode,pct_higher_education
0,101,28.10
1,147,32.57
2,151,10.48
3,153,7.56
4,155,15.49


## 3. Voter turnout — KVRES



In [127]:
dst_tableinfo("KVRES")

Table: KVRES — Valg til kommunalbestyrelser

  OMRÅDE (område) — 105 values
             000 : Hele landet
             084 : Region Hovedstaden
             101 : København
             147 : Frederiksberg
             155 : Dragør
             185 : Tårnby
             165 : Albertslund
             151 : Ballerup
             153 : Brøndby
             157 : Gentofte
    ...
             787 : Thisted
             820 : Vesthimmerlands
             851 : Aalborg
    (105 total)

  VALRES (valgresultat) — 22 values
               V : VÆLGERE
              AS : AFGIVNE STEMMER
              GS : GYLDIGE STEMMER I ALT
             GSA : A. Socialdemokratiet
             GSB : B. Radikale Venstre
             GSC : C. Det Konservative Folkeparti
             GSD : D. Nye Borgerlige
             GSF : F. SF - Socialistisk Folkeparti
             GSG : G. Veganerpartiet
             GSI : I. Liberal Alliance
    ...
              US : UGYLDIGE STEMMER I ALT
              BS : Blanke stemm

{'id': 'KVRES',
 'text': 'Valg til kommunalbestyrelser',
 'description': 'Valg til kommunalbestyrelser efter område, valgresultat og tid',
 'unit': 'Antal',
 'suppressedDataValue': '0',
 'updated': '2022-04-07T08:00:00',
 'active': True,
 'contacts': [{'name': 'Dorthe Larsen',
   'phone': '+4523498326',
   'mail': 'dla@dst.dk'}],
 'documentation': {'id': 'c7836c78-b587-461a-96b6-af5648e390e5',
  'url': 'https://www.dst.dk/statistikdokumentation/c7836c78-b587-461a-96b6-af5648e390e5'},
 'footnote': None,
 'variables': [{'id': 'OMRÅDE',
   'text': 'område',
   'elimination': True,
   'time': False,
   'map': 'Denmark_municipality_07',
   'values': [{'id': '000', 'text': 'Hele landet'},
    {'id': '084', 'text': 'Region Hovedstaden'},
    {'id': '101', 'text': 'København'},
    {'id': '147', 'text': 'Frederiksberg'},
    {'id': '155', 'text': 'Dragør'},
    {'id': '185', 'text': 'Tårnby'},
    {'id': '165', 'text': 'Albertslund'},
    {'id': '151', 'text': 'Ballerup'},
    {'id': '153', 't

### ↑ Check the output above ↑

Look for:
1. The municipality variable name (probably `OMRÅDE` or `KOMMUNE`)
2. The `VALRES` / `VALGRESULTAT` variable — find codes for "Stemmeberettigede" and "Afgivne stemmer i alt"
3. Whether `TID` includes `"2025"`

Then adjust the params in the cell below accordingly.

In [128]:


TURNOUT_PARAMS = {
    "OMRÅDE":       ["*"],                    # ← check var name
    "VALRES": ["*"],               # ← ADJUST: codes for berettigede + afgivne
    "Tid":          ["2021"],                  # ← KV25 if available, else "2021"
}

df_turnout_raw = dst_download_csv("KVRES", TURNOUT_PARAMS)
df_turnout_raw.head(10)

✓ Downloaded KVRES: 2310 rows, cols: ['OMRÅDE', 'VALRES', 'TID', 'INDHOLD']


,OMRÅDE,VALRES,TID,INDHOLD
0,Hele landet,VÆLGERE,2021,4671386
1,Hele landet,AFGIVNE STEMMER,2021,3139867
2,Hele landet,GYLDIGE STEMMER I ALT,2021,3088217
3,Hele landet,A. Socialdemokratiet,2021,877136
4,Hele landet,B. Radikale Venstre,2021,172521
5,Hele landet,C. Det Konservative Folkeparti,2021,470697
6,Hele landet,D. Nye Borgerlige,2021,110215
7,Hele landet,F. SF - Socialistisk Folkeparti,2021,235076
8,Hele landet,G. Veganerpartiet,2021,5200
9,Hele landet,I. Liberal Alliance,2021,42676


In [129]:
# Reload the canonical name → code lookup (no harm running it again)
muni_lookup = pd.read_csv("dk_municipalities.csv")[["kommune_kode", "kommune_name"]]

# Merge KVRES rows onto kommune_kode by name. inner join drops:
#   - "Hele landet"
#   - "Region Hovedstaden", etc.
#   - any other non-municipality OMRÅDE values
df_t = df_turnout_raw.merge(
    muni_lookup,
    left_on="OMRÅDE",
    right_on="kommune_name",
    how="inner",
)

print(f"After name-based join: {len(df_t)} rows, "
      f"{df_t['kommune_kode'].nunique()} unique municipalities")

# Now pivot on the actual VALRES column. Statbank returns labels like
# "Stemmeberettigede" and "Afgivne stemmer i alt", so the column we
# pivot on is just "VALRES" itself.
pivot_t = df_t.pivot_table(
    index="kommune_kode",
    columns="VALRES",
    values="INDHOLD",
    aggfunc="sum",
).reset_index()

print("Pivot columns:", list(pivot_t.columns))
pivot_t.head()

After name-based join: 2156 rows, 98 unique municipalities
Pivot columns: ['kommune_kode', 'A. Socialdemokratiet', 'AFGIVNE STEMMER', 'Andre ugyldige stemmer', 'B. Radikale Venstre', 'Blanke stemmer', 'C. Det Konservative Folkeparti', 'D. Nye Borgerlige', 'F. SF - Socialistisk Folkeparti', 'G. Veganerpartiet', 'GYLDIGE BREVSTEMMER I ALT', 'GYLDIGE STEMMER I ALT', 'I. Liberal Alliance', 'Ikke-reserverede bogstaver i alt', 'K. Kristendemokraterne', 'O. Dansk Folkeparti', 'PERSONLIGE STEMMER I ALT', 'S. Slesvigsk Parti', 'UGYLDIGE STEMMER I ALT', 'V. Venstre, Danmarks Liberale Parti', 'VÆLGERE', 'Å. Alternativet', 'Ø. Enhedslisten - De Rød-Grønne']


VALRES,kommune_kode,A. Socialdemokratiet,AFGIVNE STEMMER,Andre ugyldige stemmer,B. Radikale Venstre,Blanke stemmer,C. Det Konservative Folkeparti,D. Nye Borgerlige,F. SF - Socialistisk Folkeparti,G. Veganerpartiet,...,Ikke-reserverede bogstaver i alt,K. Kristendemokraterne,O. Dansk Folkeparti,PERSONLIGE STEMMER I ALT,S. Slesvigsk Parti,UGYLDIGE STEMMER I ALT,"V. Venstre, Danmarks Liberale Parti",VÆLGERE,Å. Alternativet,Ø. Enhedslisten - De Rød-Grønne
0,101,52874,312691,970,36688,4531,40172,6455,33824,1839,...,11552,1187,5907,181500,0,5501,23578,521387,8988,75698
1,147,10418,61747,143,5686,504,24607,484,2852,201,...,748,105,425,42136,0,647,3603,84180,482,10720
2,151,12637,25300,114,1033,300,3099,799,1758,212,...,520,0,1135,17354,0,414,1895,38403,0,1605
3,153,8104,15430,98,800,183,1146,715,941,0,...,83,0,914,10848,0,281,992,27790,0,1185
4,155,1244,8483,21,345,76,2874,172,313,0,...,2102,0,273,6423,0,97,1011,11176,0,0


In [130]:
BERETTIGEDE_COL = "VÆLGERE"
AFGIVNE_COL     = "AFGIVNE STEMMER"

print(f"Using: '{BERETTIGEDE_COL}' / '{AFGIVNE_COL}'")

turnout = pd.DataFrame({
    "kommune_kode": pivot_t["kommune_kode"],
    "voter_turnout_pct": (
        pd.to_numeric(pivot_t[AFGIVNE_COL], errors="coerce")
        / pd.to_numeric(pivot_t[BERETTIGEDE_COL], errors="coerce")
        * 100
    ).round(2),
})

print(f"\n{len(turnout)} municipalities")
print(f"Mean turnout: {turnout['voter_turnout_pct'].mean():.1f}%")
print(f"Range: {turnout['voter_turnout_pct'].min():.1f}% – "
      f"{turnout['voter_turnout_pct'].max():.1f}%")
turnout.head()

Using: 'VÆLGERE' / 'AFGIVNE STEMMER'

98 municipalities
Mean turnout: 68.9%
Range: 55.5% – 82.4%


,kommune_kode,voter_turnout_pct
0,101,59.97
1,147,73.35
2,151,65.88
3,153,55.52
4,155,75.90


In [131]:
# # ══════════════════════════════════════════════════════════════
# # ⚠ ADJUST the column names below to match pivot output     ⚠
# # ══════════════════════════════════════════════════════════════
# # Look at the pivot_t.columns output above and set:
# #   BERETTIGEDE_COL = column name for stemmeberettigede
# #   AFGIVNE_COL     = column name for afgivne stemmer

# # Try to auto-detect from column names
# cols = [str(c) for c in pivot_t.columns]
# BERETTIGEDE_COL = next((c for c in cols if "berettig" in c.lower()), None)
# AFGIVNE_COL = next((c for c in cols if "afgivne" in c.lower() or "stemmer i alt" in c.lower()), None)

# if BERETTIGEDE_COL and AFGIVNE_COL:
#     print(f"Auto-detected: '{BERETTIGEDE_COL}' / '{AFGIVNE_COL}'")
# else:
#     print(f"Could not auto-detect — set manually from: {cols}")
#     # BERETTIGEDE_COL = "..."  # ← set manually
#     # AFGIVNE_COL     = "..."  # ← set manually

# turnout = pd.DataFrame({
#     "kommune_kode": pivot_t["kommune_kode"],
#     "voter_turnout_pct": (
#         pivot_t[AFGIVNE_COL] / pivot_t[BERETTIGEDE_COL] * 100
#     ).round(2),
# })

# print(f"\n{len(turnout)} municipalities")
# print(f"Mean turnout: {turnout['voter_turnout_pct'].mean():.1f}%")
# print(f"Range: {turnout['voter_turnout_pct'].min():.1f}% – "
#       f"{turnout['voter_turnout_pct'].max():.1f}%")
# turnout.head()

In [132]:
muni = pd.read_csv(MUNI_CSV)
base = muni[["kommune_kode", "kommune_name"]].copy()

merged = (
    base
    .merge(income,    on="kommune_kode", how="left")
    .merge(education, on="kommune_kode", how="left")
    .merge(turnout,   on="kommune_kode", how="left")
)

# Coverage report
for col in ["avg_income", "pct_higher_education", "voter_turnout_pct"]:
    n = merged[col].notna().sum()
    print(f"{col}: {n}/98 municipalities")

merged.head(10)

avg_income: 98/98 municipalities
pct_higher_education: 98/98 municipalities
voter_turnout_pct: 98/98 municipalities


,kommune_kode,kommune_name,avg_income,pct_higher_education,voter_turnout_pct
0,101,København,295836,28.10,59.97
1,147,Frederiksberg,344810,32.57,73.35
2,151,Ballerup,285035,10.48,65.88
3,153,Brøndby,251169,7.56,55.52
4,155,Dragør,375847,15.49,75.90
5,157,Gentofte,570329,30.61,71.49
6,159,Gladsaxe,310421,17.67,62.43
7,161,Glostrup,280422,9.42,60.18
8,163,Herlev,291266,10.25,63.27
9,165,Albertslund,248808,8.44,61.92


In [133]:
out = merged.drop(columns=["kommune_name"])
out.to_csv(OUTPUT_CSV, index=False)
print(f"✓ Saved to {OUTPUT_CSV}")
print(f"  Shape: {out.shape}")
print(f"  Nulls: {out.isnull().sum().to_dict()}")
print(f"\nNext: run analysis_pipeline.py to generate correlations.")

✓ Saved to socioeconomic_data.csv
  Shape: (98, 4)
  Nulls: {'kommune_kode': 0, 'avg_income': 0, 'pct_higher_education': 0, 'voter_turnout_pct': 0}

Next: run analysis_pipeline.py to generate correlations.


In [134]:
pd.read_csv(OUTPUT_CSV).describe().round(1)

,kommune_kode,avg_income,pct_higher_education,voter_turnout_pct
count,98.0,98.0,98.0,98.0
mean,462.2,285976.0,8.8,68.9
std,236.6,53583.1,6.3,5.0
min,101.0,229128.0,2.6,55.5
25%,242.5,255767.0,5.0,65.9
50%,435.0,272448.5,6.7,69.1
75%,697.2,291704.0,10.2,72.2
max,860.0,570329.0,32.6,82.4


In [135]:
df["score_per_10k"].describe()

NameError: name 'df' is not defined